# Train Siamese LSTM v3 (pipeline v1.3)

**BiLSTM** 1 lớp, `bidirectional=True`, `hidden_size=255` / chiều → **pooled 510-d**, masked mean pool + L2 normalize.

- **Negative:** **online batch-hard** trên cosine (`triplet_loss_batch_hard_cosine`), không TF-IDF offline.
- **Dataset:** `QAPairDataset` (chỉ anchor + positive); `DataLoader(..., batch_size=128, drop_last=True)`.
- **Train:** `EPOCHS=50`, `PATIENCE=5`.
- **Data / embedding / artifacts:** giống trước (`data_ready_v1_3`, `shared_embedding_artifacts`, `siamese_lstm_v3_artifacts`).


In [1]:
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset


def simple_tokenize(text: str):
    return re.findall(r"\w+", str(text or "").lower().strip(), flags=re.UNICODE)


def load_artifacts(artifact_dir):
    artifact_dir = Path(artifact_dir)
    with open(artifact_dir / "tokenizer_vocab.json", "r", encoding="utf-8") as f:
        vocab = json.load(f)
    ckpt = torch.load(artifact_dir / "embedding.pt", map_location="cpu")
    stoi = vocab["stoi"]
    embedding_weight = ckpt["embedding_weight"]
    pad_idx = int(ckpt["pad_idx"])
    return stoi, embedding_weight, pad_idx


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


Device: cuda
GPU: Tesla T4


In [2]:
def detect_data_dir():
    candidates = [
        Path("data/data_ready_v1_3"),
        Path("../data/data_ready_v1_3"),
        Path("/kaggle/input/vnlegal-rag/data/data_ready_v1_3"),
        Path("/kaggle/working/vnlegal-rag/data/data_ready_v1_3"),
        Path("/kaggle/input/datasets/hngphtrn/legals-v3"),
        Path("/kaggle/input/datasets/hngphtrn/legals-v1-3"),
        Path("/kaggle/input/datasets/hngphtrn/legals_v1_3"),
    ]
    for p in candidates:
        ok = (p / "qa_train.csv").exists() and (p / "corpus_train.csv").exists()
        if p.exists() and ok:
            return p.resolve()
    raise FileNotFoundError(
        "Cannot find data_ready_v1_3 with qa_train + corpus_train. "
        "Run: python pipeline_v1.3/prepare_data.py --out-dir data/data_ready_v1_3"
    )


DATA_DIR = detect_data_dir()
ARTIFACT_DIR = Path(
    "/kaggle/working/siamese_lstm_v3_artifacts"
    if Path("/kaggle").exists()
    else "pipeline_v1.3/siamese_lstm_artifacts"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR =", DATA_DIR)
print("ARTIFACT_DIR =", ARTIFACT_DIR)


DATA_DIR = /kaggle/input/datasets/hngphtrn/legals-v3
ARTIFACT_DIR = /kaggle/working/siamese_lstm_v3_artifacts


In [3]:
qa_train = pd.read_csv(DATA_DIR / "qa_train.csv", sep="\t")
qa_val = pd.read_csv(DATA_DIR / "qa_val.csv", sep="\t")
qa_test = pd.read_csv(DATA_DIR / "qa_test.csv", sep="\t")
corpus_train = pd.read_csv(DATA_DIR / "corpus_train.csv", sep="\t")
corpus_val = pd.read_csv(DATA_DIR / "corpus_val.csv", sep="\t")
corpus_test = pd.read_csv(DATA_DIR / "corpus_test.csv", sep="\t")

print("qa_train:", qa_train.shape, "| corpus_train:", corpus_train.shape)
print("qa_val:  ", qa_val.shape, "| corpus_val:  ", corpus_val.shape)
print("qa_test: ", qa_test.shape, "| corpus_test: ", corpus_test.shape)


def pick_first_existing_col(df, candidates, df_name):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Khong tim thay cot trong {candidates} cho {df_name}. Cot: {list(df.columns)}")


QA_TEXT_COL = pick_first_existing_col(qa_train, ["question", "query", "text", "question_text"], "qa_train")
CORPUS_TEXT_COL = pick_first_existing_col(
    corpus_train,
    ["article_chunk", "article_content", "context", "text", "content", "chunk_text", "passage_text"],
    "corpus_train",
)
print("QA_TEXT_COL =", QA_TEXT_COL)
print("CORPUS_TEXT_COL =", CORPUS_TEXT_COL)


qa_train: (23311, 14) | corpus_train: (7771, 6)
qa_val:   (2841, 14) | corpus_val:   (947, 6)
qa_test:  (2991, 14) | corpus_test:  (997, 6)
QA_TEXT_COL = question
CORPUS_TEXT_COL = article_content


In [4]:
PAD = "<PAD>"
UNK = "<UNK>"
MAX_Q_LEN = 64
MAX_D_LEN = 256


def _pick_shared_embed_dir() -> Path:
    cwd = Path.cwd()
    candidates = [
        Path("/kaggle/input/datasets/hngphtrn/legal-embedding-v3"),
        cwd / "pipeline_v1.3" / "shared_embedding_artifacts",
        cwd / "shared_embedding_artifacts",
        Path("../pipeline_v1.3/shared_embedding_artifacts"),
    ]
    for p in candidates:
        if p.exists() and (p / "embedding.pt").is_file() and (p / "tokenizer_vocab.json").is_file():
            return p
    raise FileNotFoundError(
        "No shared embedding. Run: python pipeline_v1.3/build_shared_embedding.py"
    )


SHARED_EMBED_DIR = _pick_shared_embed_dir()
stoi, embedding_weight, pad_idx = load_artifacts(SHARED_EMBED_DIR)
itos = {i: w for w, i in stoi.items()}
MAX_VOCAB = len(stoi)

vocab_meta = json.loads((SHARED_EMBED_DIR / "tokenizer_vocab.json").read_text(encoding="utf-8"))
build_meta = vocab_meta.get("build_metadata", {})
if not build_meta.get("corpus_path"):
    print("WARNING: vocab build_metadata may miss corpus_path; rebuild embedding if needed.")


def encode_text(text, max_len):
    toks = simple_tokenize(text)
    ids = [stoi.get(t, stoi["<UNK>"]) for t in toks[:max_len]]
    if len(ids) < max_len:
        ids += [stoi["<PAD>"]] * (max_len - len(ids))
    return ids


print("Loaded shared vocab size:", len(stoi), "| embedding shape:", tuple(embedding_weight.shape))
print("SHARED_EMBED_DIR =", SHARED_EMBED_DIR)
print("Built from:", build_meta.get("qa_path", "?"), "+", build_meta.get("corpus_path", "?"))


Loaded shared vocab size: 6227 | embedding shape: (6227, 200)
SHARED_EMBED_DIR = /kaggle/input/datasets/hngphtrn/legal-embedding-v3
Built from: C:\Users\hung\Documents\GitHub\vnlegal-rag\data\data_ready_v1_3\qa_train.csv + C:\Users\hung\Documents\GitHub\vnlegal-rag\data\data_ready_v1_3\corpus_train.csv


In [5]:
corpus_full = pd.concat([corpus_train, corpus_val, corpus_test], ignore_index=True)
corpus_full_map = corpus_full.set_index("passage_id").to_dict(orient="index")

corpus_train_map = corpus_train.set_index("passage_id").to_dict(orient="index")
corpus_val_map = corpus_val.set_index("passage_id").to_dict(orient="index")
corpus_test_map = corpus_test.set_index("passage_id").to_dict(orient="index")


## Training pairs + online batch-hard negatives

Mỗi batch: encode `z_q`, `z_d+`; hard negative = **document positive của cặp khác** trong batch có `cos(q_i, d_j+)` cao nhất (`j≠i`). Cần `drop_last=True` và `batch_size≥2`.


In [6]:
class QAPairDataset(Dataset):
    def __init__(self, qa_df, corpus_map):
        self.qa_df = qa_df.reset_index(drop=True)
        self.corpus_map = corpus_map

    def __len__(self):
        return len(self.qa_df)

    def __getitem__(self, idx):
        row = self.qa_df.iloc[idx]
        q = str(row[QA_TEXT_COL])
        pid = row["passage_id"]
        doc = self.corpus_map[pid]
        return {
            "anchor": torch.tensor(encode_text(q, MAX_Q_LEN), dtype=torch.long),
            "positive": torch.tensor(encode_text(str(doc[CORPUS_TEXT_COL]), MAX_D_LEN), dtype=torch.long),
        }


import sys

BATCH_SIZE = 128
NUM_WORKERS = 0 if sys.platform == "win32" else 2
train_ds = QAPairDataset(qa_train, corpus_train_map)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
    drop_last=True,
)
print("Train pairs:", len(train_ds), "| batches/epoch:", len(train_loader), "| BATCH_SIZE=", BATCH_SIZE, "drop_last=True")


Train pairs: 23311 | batches/epoch: 182 | BATCH_SIZE= 128 drop_last=True


In [7]:
class LSTMEncoder(nn.Module):
    """BiLSTM + masked mean pool. `hidden_size` = hidden mỗi chiều; output dim = 2 * hidden_size."""

    def __init__(self, vocab_size, embed_dim=200, hidden_size=255, num_layers=1, dropout=0.3, pad_idx=0, embedding_weight=None):
        super().__init__()
        if embedding_weight is None:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        else:
            self.embedding = nn.Embedding.from_pretrained(embedding_weight, freeze=False, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.pad_idx = pad_idx
        self.hidden_size = hidden_size

    def forward(self, x):
        mask = (x != self.pad_idx).float().unsqueeze(-1)
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        summed = (out * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        pooled = summed / denom
        return F.normalize(pooled, p=2, dim=1)


class SiameseLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_size=255, num_layers=1, dropout=0.3, pad_idx=0, embedding_weight=None):
        super().__init__()
        self.encoder = LSTMEncoder(vocab_size, embed_dim, hidden_size, num_layers, dropout, pad_idx, embedding_weight=embedding_weight)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, a, p):
        return self.encode(a), self.encode(p)


In [8]:
EMBED_DIM = int(embedding_weight.shape[1])
HIDDEN_SIZE = 255
NUM_LAYERS = 1
ENCODER_DROPOUT = 0.3
ENCODE_DIM = 2 * HIDDEN_SIZE

print(f"BiLSTM: num_layers={NUM_LAYERS}, bidirectional=True, hidden/đirection={HIDDEN_SIZE}, pooled_dim={ENCODE_DIM}")

model = SiameseLSTM(
    vocab_size=len(stoi),
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=ENCODER_DROPOUT,
    pad_idx=stoi["<PAD>"],
    embedding_weight=embedding_weight,
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params (full model): {trainable_params:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
scaler = GradScaler("cuda", enabled=torch.cuda.is_available())
margin = 0.3


def triplet_loss_batch_hard_cosine(za, zp, margin=0.3):
    """za, zp: (B,D) L2-normalized. Hard neg = max_{j≠i} cos(zq_i, zd_j+)."""
    sim = torch.matmul(za, zp.transpose(0, 1))
    b = sim.size(0)
    if b < 2:
        return sim.new_zeros(())
    mask = torch.eye(b, device=sim.device, dtype=torch.bool)
    neg_sim = sim.masked_fill(mask, float("-inf"))
    hard_neg, _ = neg_sim.max(dim=1)
    pos_sim = sim.diag()
    return F.relu(margin - pos_sim + hard_neg).mean()


BiLSTM: num_layers=1, bidirectional=True, hidden/đirection=255, pooled_dim=510
Trainable params (full model): 2,177,680


In [9]:
def train_one_epoch(model, loader):
    model.train()
    total = 0.0
    for batch in tqdm(loader, leave=False):
        a = batch["anchor"].to(device)
        p = batch["positive"].to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            za, zp = model(a, p)
            loss = triplet_loss_batch_hard_cosine(za, zp, margin=margin)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += float(loss.item())
    return total / max(1, len(loader))


In [10]:
@torch.no_grad()
def encode_passage_matrix(model, corpus_df, batch_size=64):
    model.eval()
    pids = corpus_df["passage_id"].tolist()
    vecs = []
    for i in range(0, len(pids), batch_size):
        texts = corpus_df.iloc[i : i + batch_size][CORPUS_TEXT_COL].astype(str).tolist()
        ids = torch.tensor([encode_text(t, MAX_D_LEN) for t in texts], dtype=torch.long, device=device)
        z = model.encode(ids)
        vecs.append(z.cpu())
    return pids, torch.cat(vecs, dim=0)


@torch.no_grad()
def evaluate_retrieval(model, qa_df, corpus_df, topk=(1, 3, 5), max_queries=1500):
    model.eval()
    pids, pmat = encode_passage_matrix(model, corpus_df, batch_size=64)
    pid_to_idx = {p: i for i, p in enumerate(pids)}
    qa_eval = qa_df if (max_queries is None or len(qa_df) <= max_queries) else qa_df.sample(max_queries, random_state=42)
    hits = {k: 0 for k in topk}
    rr_sum = 0.0
    for _, row in tqdm(qa_eval.iterrows(), total=len(qa_eval), leave=False):
        q_ids = torch.tensor([encode_text(str(row[QA_TEXT_COL]), MAX_Q_LEN)], dtype=torch.long, device=device)
        q_vec = model.encode(q_ids).cpu()
        scores = torch.matmul(q_vec, pmat.T).squeeze(0)
        order = torch.argsort(scores, descending=True)
        true_pid = row["passage_id"]
        true_idx = pid_to_idx.get(true_pid, None)
        if true_idx is None:
            continue
        rank_pos = (order == true_idx).nonzero(as_tuple=True)[0]
        if len(rank_pos) == 0:
            continue
        rank = int(rank_pos.item()) + 1
        rr_sum += 1.0 / rank
        for k in topk:
            if rank <= k:
                hits[k] += 1
    n = len(qa_eval)
    metrics = {f"Recall@{k}": hits[k] / max(1, n) for k in topk}
    metrics["MRR"] = rr_sum / max(1, n)
    return metrics


In [11]:
EPOCHS = 50
PATIENCE = 5
MIN_DELTA = 1e-4
EMBED_FREEZE_EPOCHS = int(globals().get("EMBED_FREEZE_EPOCHS", 0))
embedding_params = list(globals().get("embedding_params", model.encoder.embedding.parameters()))
best_score = -1.0
best_epoch = 0
wait = 0
best_path = ARTIFACT_DIR / "siamese_lstm_traditional_cosine_best.pt"
history = []

for epoch in range(1, EPOCHS + 1):
    freeze_embedding = epoch <= EMBED_FREEZE_EPOCHS
    for p in embedding_params:
        p.requires_grad = not freeze_embedding

    tr_loss = train_one_epoch(model, train_loader)
    val_metrics = evaluate_retrieval(model, qa_val, corpus_full, topk=(1, 3, 5), max_queries=1500)
    val_score = val_metrics["MRR"]
    scheduler.step(val_score)

    row = {"epoch": epoch, "train_loss": tr_loss, "freeze_embedding": freeze_embedding, **val_metrics}
    history.append(row)
    phase = "frozen" if freeze_embedding else "finetune"
    print(
        f"Epoch {epoch:02d} [{phase}] | loss={tr_loss:.4f} | MRR={val_metrics['MRR']:.4f} | "
        f"R@1={val_metrics['Recall@1']:.4f} | R@5={val_metrics['Recall@5']:.4f}"
    )

    if val_score > best_score + MIN_DELTA:
        best_score = val_score
        best_epoch = epoch
        wait = 0
        torch.save(model.state_dict(), best_path)
        print(f"  -> Saved best (MRR={best_score:.4f})")
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stop epoch {epoch}. Best: {best_epoch} MRR={best_score:.4f}")
            break

print(f"Best val MRR: {best_score:.4f} (epoch {best_epoch})")
print("Checkpoint:", best_path)


  0%|          | 0/182 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch 01 [finetune] | loss=0.2124 | MRR=0.5146 | R@1=0.3713 | R@5=0.7007
  -> Saved best (MRR=0.5146)


  0%|          | 0/182 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

...

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch 49 [finetune] | loss=0.0193 | MRR=0.6888 | R@1=0.5460 | R@5=0.8620


  0%|          | 0/182 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch 50 [finetune] | loss=0.0188 | MRR=0.6929 | R@1=0.5520 | R@5=0.8653
Best val MRR: 0.6959 (epoch 48)
Checkpoint: /kaggle/working/siamese_lstm_v3_artifacts/siamese_lstm_traditional_cosine_best.pt


In [12]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_metrics = evaluate_retrieval(model, qa_test, corpus_test, topk=(1, 3, 5, 10), max_queries=None)
print("Test metrics:", test_metrics)


  0%|          | 0/2991 [00:00<?, ?it/s]

Test metrics: {'Recall@1': 0.7295218990304246, 'Recall@3': 0.8548980274155801, 'Recall@5': 0.8849882982280174, 'Recall@10': 0.9284520227348713, 'MRR': 0.8012353613331203}


In [13]:
with open(ARTIFACT_DIR / "tokenizer_vocab.json", "w", encoding="utf-8") as f:
    json.dump({"stoi": stoi, "itos": {str(k): v for k, v in itos.items()}}, f, ensure_ascii=False)

with open(ARTIFACT_DIR / "siamese_meta.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "distance_metric": "cosine",
            "encoder": "bilstm_traditional_cosine",
            "pipeline": "v1.3",
            "data_dir": str(DATA_DIR),
            "shared_embed_dir": str(SHARED_EMBED_DIR),
            "max_q_len": MAX_Q_LEN,
            "max_d_len": MAX_D_LEN,
            "margin": margin,
            "embed_dim": EMBED_DIM,
            "bidirectional": True,
            "hidden_size_per_direction": HIDDEN_SIZE,
            "pooled_dim": ENCODE_DIM,
            "num_layers": NUM_LAYERS,
            "dropout": ENCODER_DROPOUT,
            "trainable_params": trainable_params,
            "negatives": "batch_hard_online_cosine",
            "batch_size": BATCH_SIZE,
            "drop_last": True,
            "epochs": EPOCHS,
            "patience": PATIENCE,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame(history).to_csv(ARTIFACT_DIR / "train_history.csv", index=False)
print("Saved artifacts at:", ARTIFACT_DIR)


Saved artifacts at: /kaggle/working/siamese_lstm_v3_artifacts
